In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup, Comment
from io import StringIO
import os
from datetime import datetime
import pytz

# Use today's date in Eastern Time for filenames
eastern = pytz.timezone("US/Eastern")
today_str = datetime.now(eastern).date().isoformat()

# === Base path: resolve up one directory and into /data ===
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), "../data"))

# Tables to scrape from Baseball Reference
tables_to_scrape = [
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml",
        "table_id": "teams_standard_pitching",
        "folder": "team_pitching",
        "filename_prefix": "team_pitching"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml",
        "table_id": "players_standard_pitching",
        "folder": "player_pitching",
        "filename_prefix": "player_pitching"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml",
        "table_id": "teams_standard_batting",
        "folder": "team_batting",
        "filename_prefix": "team_batting"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml",
        "table_id": "players_standard_batting",
        "folder": "player_batting",
        "filename_prefix": "player_batting"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/majors/2025.shtml",
        "table_id": "team_output",
        "folder": "team_war",
        "filename_prefix": "team_war"
    },
    {
        "url": "https://www.baseball-reference.com/leagues/MLB-standings.shtml",
        "table_id": "expanded_standings_overall",
        "folder": "standings",
        "filename_prefix": "standings"
    }
]

def scrape_table(url, table_id):
    print(f"📡 Fetching data from: {url}")
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')

    table = soup.find('table', {'id': table_id})

    if table is None:
        comments = soup.find_all(string=lambda text: isinstance(text, Comment))
        for comment in comments:
            if table_id in comment:
                comment_soup = BeautifulSoup(comment, 'html.parser')
                table = comment_soup.find('table', {'id': table_id})
                if table:
                    break

    if table is None:
        print(f"❌ Table with ID '{table_id}' not found.")
        return None

    df = pd.read_html(StringIO(str(table)))[0]
    if 'Rk' in df.columns:
        df = df[df['Rk'] != 'Rk']
        df.reset_index(drop=True, inplace=True)
    return df

def save_dataframe(df, subfolder, filename_prefix):
    full_folder = os.path.join(BASE_DIR, subfolder)
    os.makedirs(full_folder, exist_ok=True)
    filename = f"{filename_prefix}_{today_str}.csv"
    filepath = os.path.join(full_folder, filename)
    df.to_csv(filepath, index=False)
    print(f"✅ Saved: {filepath}")

# Run all scrapes
for table_info in tables_to_scrape:
    df = scrape_table(table_info["url"], table_info["table_id"])
    if df is not None:
        save_dataframe(df, table_info["folder"], table_info["filename_prefix"])


📡 Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml
✅ Saved: /workspaces/MLB_Model/data/team_pitching/team_pitching_2025-06-04.csv
📡 Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-pitching.shtml
✅ Saved: /workspaces/MLB_Model/data/player_pitching/player_pitching_2025-06-04.csv
📡 Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml
✅ Saved: /workspaces/MLB_Model/data/team_batting/team_batting_2025-06-04.csv
📡 Fetching data from: https://www.baseball-reference.com/leagues/majors/2025-standard-batting.shtml
✅ Saved: /workspaces/MLB_Model/data/player_batting/player_batting_2025-06-04.csv
📡 Fetching data from: https://www.baseball-reference.com/leagues/majors/2025.shtml
✅ Saved: /workspaces/MLB_Model/data/team_war/team_war_2025-06-04.csv
📡 Fetching data from: https://www.baseball-reference.com/leagues/MLB-standings.shtml
✅ Saved: /workspaces/MLB_Model/data/stand